# N4 solution. Guided activity of Notebook 4, solved

**Predicting Flash Floods in Working Landscapes**, Module 3, Event based evaluation and forecast skill
CUAHSI Water Science Exchange 2026

This short notebook holds the completed activity of Notebook 4 (section 7), with its outputs. It rebuilds only what the activity needs, so it runs on its own in a few seconds. The full notebook with every section executed is `04_are_the_probabilities_trustworthy_solution.ipynb`.

The task. A deterministic forecast is a probability forecast that only says 0 or 1. Score the control run that way and compare it with the raw and the calibrated ensembles.

## 0. Setup

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AHWALab/Water-Science-Exchange-Workshop.git"
BRANCH = "module-3-evaluation"

# In Colab the helper files are not here yet. Copy the branch (public, no account needed).
if not Path("hymod_tools.py").exists():
    if not Path("Water-Science-Exchange-Workshop").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL, "Water-Science-Exchange-Workshop"], check=True)
    os.chdir("Water-Science-Exchange-Workshop/notebooks")
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

Working folder: /home/claude/m3/notebooks


## 1. What the activity needs from Notebook 4

The data, the threshold, the control run, the raw ensemble and its Brier scores (section 2), and the calibrated ensemble of section 6 with its Brier skill score.

In [2]:
import numpy as np
import hymod_tools as h

PP, PET, QQ = h.load_leaf_river("LeafCatch.dat")      # 40 years of daily data, mm per day
N_EVAL = 365 * 5                                        # the five year evaluation window
Qobs = QQ[:N_EVAL]
flood_th = h.flood_threshold(QQ, fraction=0.25)
Qctrl = h.hymod01(PP[:N_EVAL], PET[:N_EVAL], h.CONTROL_PARS)["Q"]      # the control run
ensemble, _ = h.generate_ensemble(PP, PET, N_EVAL, Nens=100, seed=7)   # the raw ensemble
obs_events = (Qobs >= flood_th).astype(int)
prob_exceed = (ensemble >= flood_th).mean(axis=0)
climatology = obs_events.mean()                          # the base rate, the reference forecast

# Brier score of the raw ensemble and of climatology, and the skill score (section 2).
BS = np.mean((prob_exceed - obs_events) ** 2)
BS_clim = np.mean((climatology - obs_events) ** 2)
BSS = 1.0 - BS / BS_clim

# The calibrated ensemble of section 6, rainfall centred on 1.0 and parameters around the control.
def generate_calibrated_ensemble(seed=7, Nens=100):
    """A better behaved ensemble. Rainfall centred at 1.0, parameters around the control values."""
    rng = np.random.default_rng(seed)
    cp = h.CONTROL_PARS
    ens = np.zeros((Nens, N_EVAL))
    for i in range(Nens):
        mult = max(0.05, rng.normal(1.0, 0.18))              # rainfall centred on the observed depth
        pars = {"Huz": float(np.clip(cp["Huz"] * rng.uniform(0.6, 1.4), 5, 500)),
                "B": float(np.clip(cp["B"] * rng.uniform(0.6, 1.4), 0.05, 2)),
                "Alp": float(np.clip(cp["Alp"] * rng.uniform(0.7, 1.3), 0.02, 0.98)),
                "Nq": cp["Nq"],
                "Kq": float(np.clip(cp["Kq"] * rng.uniform(0.6, 1.4), 0.05, 0.98)),
                "Ks": float(np.clip(cp["Ks"] * rng.uniform(0.5, 1.5), 0.0005, 0.2))}
        ens[i] = h.hymod01(PP[:N_EVAL] * mult, PET[:N_EVAL], pars)["Q"]
    return ens

ens_cal = generate_calibrated_ensemble(seed=7)
prob_cal = (ens_cal >= flood_th).mean(axis=0)

ens_cal = generate_calibrated_ensemble(seed=7)
prob_cal = (ens_cal >= flood_th).mean(axis=0)
BS_cal = np.mean((prob_cal - obs_events) ** 2)
BSS_cal = 1 - BS_cal / BS_clim
print(f"BSS raw ensemble {BSS:+.3f}, calibrated ensemble {BSS_cal:+.3f}")

BSS raw ensemble -0.008, calibrated ensemble +0.521


## 2. The activity, solved

In [3]:
#################### Your Code Goes Here #########################
# Step 1. The control run as a probability. 1 on days when Qctrl is at or above the threshold, else 0.
prob_ctrl = (Qctrl >= flood_th).astype(float)

# Step 2. Its Brier score and Brier skill score (same formulas as section 2).
BS_ctrl = np.mean((prob_ctrl - obs_events) ** 2)
BSS_ctrl = 1 - BS_ctrl / BS_clim

# Step 3. Print the three Brier skill scores side by side, control, raw ensemble, calibrated ensemble.
print(f"BSS control {BSS_ctrl:+.3f}   raw ensemble {BSS:+.3f}   calibrated ensemble {BSS_cal:+.3f}")
###################################################################

BSS control +0.165   raw ensemble -0.008   calibrated ensemble +0.521


Reading the result. The control run beats the raw ensemble on the Brier score because the raw ensemble is biased low and never commits. It does not beat the calibrated ensemble, and a 0 or 1 forecast cannot express uncertainty at all, so its reliability diagram is two points and its CRPS is simply the absolute error. The ensemble is the better forecast once it is calibrated.